# Full Time-Series Builder (Research-Aligned)

**Purpose:** Build the production dataset `alibaba_timeseries_full.csv` from the raw Alibaba DLRM trace using the same method used in the sample builder, while explicitly documenting research parameters and rationale.


In [ ]:
import os
import numpy as np
import pandas as pd


In [ ]:
# Research-aligned configuration
# Full build defaults:
SAMPLE_MODE = False
SAMPLE_N_APPS = None
MAX_TRACE_SECONDS = None

DATA_PATH = 'Alibaba/disaggregated_DLRM_trace.csv'
OUTPUT_PATH = 'alibaba_timeseries_full.csv'

# Time-series construction parameters (used in preprocessing):
INTERVAL = 300
HISTORY_LENGTH = 24
FORECAST_STEPS = 6

# Downstream model/evaluation parameters (documented for consistency):
N_CHUNKS = 4
EWC_LAMBDA = 100
REPLAY_MEMORY = 500
REPLAY_RATIO = 0.3
ROLL_WINDOW = 12
SLA_TOL = 0.15

TEMPORAL_FEATURES = [
    'cpu_demand', 'cpu_diff', 'cpu_roll_mean', 'cpu_roll_std',
    'cpu_roll_min', 'cpu_roll_max', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]
STATIC_FEATURES = [
    'gpu_request_mean', 'memory_request_mean', 'rdma_request_mean',
    'role_hn_fraction', 'instance_count', 'max_instance_per_node'
]

print('=' * 72)
print('Full time-series build configuration')
print(f'SAMPLE_MODE      : {SAMPLE_MODE}')
print(f'DATA_PATH        : {DATA_PATH}')
print(f'OUTPUT_PATH      : {OUTPUT_PATH}')
print(f'INTERVAL         : {INTERVAL}s')
print(f'HISTORY_LENGTH   : {HISTORY_LENGTH} steps')
print(f'FORECAST_STEPS   : {FORECAST_STEPS} steps')
print(f'N_CHUNKS         : {N_CHUNKS}')
print(f'EWC_LAMBDA       : {EWC_LAMBDA}')
print(f'REPLAY_MEMORY    : {REPLAY_MEMORY}')
print(f'REPLAY_RATIO     : {REPLAY_RATIO}')
print(f'ROLL_WINDOW      : {ROLL_WINDOW} (1 hour with 5-min bins)')
print(f'SLA_TOL          : {SLA_TOL}')
print('=' * 72)


## Parameter rationale (from proposal + implementation + evaluation guide)

- `INTERVAL=300` (5 min): proposal Section 5.2.1 states five-minute resampling for workload time-series.
- `HISTORY_LENGTH=24`: 24 x 5 min = 2-hour look-back window used in the hybrid model branch input.
- `FORECAST_STEPS=6`: 6 x 5 min = 30-minute ahead CPU demand prediction target.
- `N_CHUNKS=4`: continual learning split used to evaluate adaptation and forgetting across sequential workload periods.
- `EWC_LAMBDA=100`: default Fisher penalty strength for balancing plasticity and stability.
- `REPLAY_MEMORY=500`, `REPLAY_RATIO=0.3`: replay configuration for retaining prior workload knowledge while learning new chunks.
- `ROLL_WINDOW=12`: one-hour rolling statistics in downstream feature engineering.
- `SLA_TOL=0.15`: evaluation tolerance (15%) for under/over-provisioning checks in SLA analysis.

The raw event-based trace is converted to regular bins so the LSTM can consume aligned sequences and the MLP can consume per-app static/profile features. This directly supports the research objective of proactive CPU autoscaling with continual learning.


In [ ]:
# Load raw Alibaba DLRM trace
df = pd.read_csv(DATA_PATH)

print(f'Raw dataset shape : {df.shape}')
print(f'Columns           : {df.columns.tolist()}')
print()
print('Role distribution (CN = CPU Node, HN = Heterogeneous Node):')
print(df['role'].value_counts())
print()
print('Missing value summary:')
print(df.isnull().sum())
df.head(3)


In [ ]:
# Build 5-minute time-series from event-based records
if os.path.exists(OUTPUT_PATH):
    print(f'Found existing file: {OUTPUT_PATH}')
    print('Delete it to force a rebuild, or continue to use cached output.')
    ts_df = pd.read_csv(OUTPUT_PATH)
    print(f'Loaded {len(ts_df):,} rows, {ts_df["app_name"].nunique()} apps.')
else:
    print('Building full time-series ...')

    # Step 1: handle missing timestamps
    df_work = df.copy()
    df_work['creation_time'] = df_work['creation_time'].fillna(0)
    full_trace_end = df_work['deletion_time'].dropna().max()
    df_work['deletion_time'] = df_work['deletion_time'].fillna(full_trace_end)

    # Step 2: optional clipping (disabled in full mode by default)
    if SAMPLE_MODE and MAX_TRACE_SECONDS is not None:
        trace_end = float(MAX_TRACE_SECONDS)
        df_work = df_work[df_work['creation_time'] < trace_end].copy()
        df_work['deletion_time'] = df_work['deletion_time'].clip(upper=trace_end)
    else:
        trace_end = float(full_trace_end)

    # Step 3: encode and clean
    df_work['role_encoded'] = (df_work['role'] == 'HN').astype(int)
    df_work['max_instance_per_node'] = df_work['max_instance_per_node'].replace(-1, np.nan)

    n_bins = int(trace_end / INTERVAL)
    print(f'Trace window : 0 - {trace_end:.0f}s ({trace_end / 3600:.1f}h)')
    print(f'Time bins    : {n_bins} ({INTERVAL}s each)')
    print(f'Instances    : {len(df_work):,}')

    # Step 4: app selection
    if SAMPLE_MODE and SAMPLE_N_APPS:
        top_apps = (df_work.groupby('app_name')['instance_sn']
                           .count().nlargest(SAMPLE_N_APPS).index)
        df_proc = df_work[df_work['app_name'].isin(top_apps)].copy()
        print(f'Sample       : {len(top_apps)} apps, {len(df_proc):,} instances')
    else:
        df_proc = df_work.copy()
        print(f'Full dataset : {len(df_proc):,} instances')

    # Step 5: per-bin vectorized aggregation
    time_steps = np.arange(0, trace_end, INTERVAL)
    agg_records = []
    ct = df_proc['creation_time'].to_numpy()
    dt = df_proc['deletion_time'].to_numpy()

    for idx, t in enumerate(time_steps):
        if idx % 50 == 0:
            print(f'  Bin {idx + 1}/{len(time_steps)} ({t / 3600:.1f}h) ...')
        active = df_proc[(ct <= t) & (dt > t)]
        if active.empty:
            continue

        agg = active.groupby('app_name', sort=False).agg(
            cpu_demand=('cpu_request', 'sum'),
            gpu_request_mean=('gpu_request', 'mean'),
            memory_request_mean=('memory_request', 'mean'),
            rdma_request_mean=('rdma_request', 'mean'),
            role_hn_fraction=('role_encoded', 'mean'),
            instance_count=('cpu_request', 'count'),
            max_instance_per_node=('max_instance_per_node', 'mean'),
        ).assign(timestamp=t)
        agg_records.append(agg)

    ts_df = pd.concat(agg_records).reset_index()
    ts_df = ts_df.sort_values(['app_name', 'timestamp']).reset_index(drop=True)
    ts_df['max_instance_per_node'] = ts_df['max_instance_per_node'].fillna(0)

    # Step 6: remove apps with too few steps for sequence setup
    min_steps = HISTORY_LENGTH + FORECAST_STEPS + 10
    app_counts = ts_df.groupby('app_name')['timestamp'].count()
    valid_apps = app_counts[app_counts >= min_steps].index
    ts_df = ts_df[ts_df['app_name'].isin(valid_apps)].reset_index(drop=True)

    # Step 7: save
    ts_df.to_csv(OUTPUT_PATH, index=False)
    print(f'Saved -> {OUTPUT_PATH}')


In [ ]:
print('=' * 72)
print(f'Output file  : {OUTPUT_PATH}')
print(f'Rows         : {len(ts_df):,}')
print(f'Apps         : {ts_df["app_name"].nunique()}')
print(f'Columns      : {ts_df.columns.tolist()}')
print(f'Time range   : {ts_df["timestamp"].min():.0f}s - {ts_df["timestamp"].max():.0f}s')
print('=' * 72)
print()
print('CPU demand statistics:')
print(ts_df['cpu_demand'].describe().round(3))
print()
print('Set this in research notebook:')
print(f"  PREPROCESSED_PATH = '{OUTPUT_PATH}'")


## Python script equivalent

This notebook has a script counterpart:

- `build_timeseries_full.py`

Run from terminal:

```bash
python build_timeseries_full.py
```

Sample-mode run:

```bash
python build_timeseries_full.py --sample-mode --sample-n-apps 10 --max-trace-seconds 86400 --output-path alibaba_timeseries_sample.csv
```
